# Neuron set examples

Examples for how to use different types of neuron sets.

In [ ]:
import obi_one as obi
from pathlib import Path

### __Initialization:__ Loading a circuit

In [ ]:
circuit_name = "N_10__top_nodes_dim6"

circuit_path_prefix = Path("../data/tiny_circuits")
matrix_path_prefix = Path("../data/connectivity_matrices")  # OPTIONAL: Connectivity matrix path only required for some of the node sets in this example notebook; can be set to None

circuit_path = circuit_path_prefix / circuit_name / "circuit_config.json"
if matrix_path_prefix is None:
    matrix_path = None
else:
    matrix_path = matrix_path_prefix / circuit_name / "connectivity_matrix.h5"
circuit = obi.Circuit(name=circuit_name, path=str(circuit_path), matrix_path=str(matrix_path))

print(f"Circuit '{circuit}' with {circuit.sonata_circuit.nodes[circuit.default_population_name].size} neurons and {circuit.sonata_circuit.edges[circuit.default_edge_population_name].size} synapses")
print(f"Node populations: '{circuit.sonata_circuit.nodes.population_names}'")
print(f"Edge populations: '{circuit.sonata_circuit.edges.population_names}'")

### **New neuron set examples**

The new neuron sets have `population` as a field (not a method argument).
They return:
- `dict[str, list[int]]` from `get_neuron_ids`: Neuron IDs by population
- `tuple[dict|list, dict]` from `get_node_set_definition`: Base expression + additional definitions in case of compound expression

In [ ]:
from obi_one.scientific.blocks.neuron_sets_2.combined import BiophysicalCombinedNeuronSet, SetOperation
from obi_one.scientific.blocks.neuron_sets_2.id import BiophysicalIDNeuronSet
from obi_one.scientific.blocks.neuron_sets_2.population import PopulationNeuronSet, BiophysicalPopulationNeuronSet
from obi_one.scientific.blocks.neuron_sets_2.predefined import PredefinedNeuronSet, PredefinedPopulationNeuronSet, PredefinedVirtualPopulationNeuronSet
from obi_one.scientific.blocks.neuron_sets_2.property import PropertyNeuronSet
from obi_one.scientific.blocks.neuron_sets_2.property import NeuronPropertyFilter
from obi_one.scientific.blocks.neuron_sets_2.specific import AllNeurons, AllBiophysicalNeurons
from obi_one.scientific.unions.unions_neuron_sets_2 import BiophysicalNeuronSetReference
from obi_one.core.tuple import NamedTuple

#### (a) PopulationNeuronSet — sample % from a population

In [ ]:
nset = PopulationNeuronSet(population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=1)

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"PopulationNeuronSet (50%)")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition: {nset_def}")

#### (b) PopulationNeuronSet — full population (symbolic expression)

In [ ]:
nset = BiophysicalPopulationNeuronSet(population="S1nonbarrel_neurons")

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"BiophysicalPopulationNeuronSet (100%)")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition (symbolic): {nset_def}")

#### (c) PredefinedNeuronSet — use an existing node set (multi-population)

In [ ]:
nset = PredefinedNeuronSet(node_set="AllPopulations")
nset.set_block_name("nset")

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"PredefinedNeuronSet '{nset.node_set}'")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition (symbolic): {nset_def} {combined}")

# Force resolve IDs
nset_def_resolved, combined_resolved = nset.get_node_set_definition(circuit, force_resolve_ids=True)
print(f"  Node set definition (resolved): {nset_def_resolved} {combined_resolved}")

#### (d) PredefinedPopulationNeuronSet — predefined node set resolved in one population (with sampling)

In [ ]:
nset = PredefinedPopulationNeuronSet(
    node_set="Layer6", population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=42
)

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"PredefinedPopulationNeuronSet 'Layer6' (50% sample)")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition: {nset_def}")

In [ ]:
# Wrong population type
nset = PredefinedVirtualPopulationNeuronSet(
    node_set="Layer6", population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=42
)

# This should raise a ValueError about wrong population type
try:
    nset_def, combined = nset.get_node_set_definition(circuit)
except ValueError as e:
    print(f"Caught expected error: {e}")

#### (e) IDNeuronSet — by neuron IDs

In [ ]:
nset = BiophysicalIDNeuronSet(
    population="S1nonbarrel_neurons",
    neuron_ids=NamedTuple(name="my_ids", elements=(0, 2, 5, 8)),
)

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"BiophysicalIDNeuronSet")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition: {nset_def}")

#### (f) PropertyNeuronSet — by neuron properties

In [ ]:
nset = PropertyNeuronSet(
    population="S1nonbarrel_neurons",
    property_filter=NeuronPropertyFilter(filter_dict={"layer": ["6"], "synapse_class": ["EXC"]}),
)

ids = nset.get_neuron_ids(circuit)
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"PropertyNeuronSet (layer=6, synapse_class=EXC)")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")
print(f"  Node set definition: {nset_def}")

# With sampling
nset_sampled = PropertyNeuronSet(
    population="S1nonbarrel_neurons",
    property_filter=NeuronPropertyFilter(filter_dict={"layer": ["6"], "synapse_class": ["EXC"]}),
    sample_percentage=50,
    sample_seed=7,
)
nset_sampled.set_block_name("prop_nset_sampled")
ids_sampled = nset_sampled.get_neuron_ids(circuit)
print(f"\n  With 50% sampling: {ids_sampled}")

#### (g) CombinedNeuronSet — union, intersection, difference

In [ ]:
# Create two base neuron sets to combine
nset_a = PredefinedPopulationNeuronSet(node_set="All", population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=0)
nset_a.set_block_name("nset_a")

nset_b = PredefinedPopulationNeuronSet(node_set="All", population="S1nonbarrel_neurons", sample_percentage=50, sample_seed=1)
nset_b.set_block_name("nset_b")

# Wrap in BlockReferences (as required by the combined neuron set fields)
ref_a = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_a")
ref_a.block = nset_a
ref_b = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_b")
ref_b.block = nset_b

print(f"Set A (50%, seed 0): {nset_a.get_neuron_ids(circuit)}")
print(f"Set B (50%, seed 1): {nset_b.get_neuron_ids(circuit)}")

In [ ]:
# Union
combined_union = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=ref_b, operation=SetOperation.UNION
)
combined_union.set_block_name("union_ab")
print(f"Union: {combined_union.get_neuron_ids(circuit)}")
print(f"  Populations: {combined_union.get_populations(circuit)}")
print(f"  Neuron IDs: {combined_union.get_neuron_ids(circuit)}")
print(f"  Node set def: {combined_union.get_node_set_definition(circuit)}")

In [ ]:
# Intersection
combined_intersect = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=ref_b, operation=SetOperation.INTERSECT
)
combined_intersect.set_block_name("intersect_ab")
print(f"Intersection: {combined_intersect.get_neuron_ids(circuit)}")
print(f"  Populations: {combined_intersect.get_populations(circuit)}")
print(f"  Neuron IDs: {combined_intersect.get_neuron_ids(circuit)}")
print(f"  Node set def: {combined_intersect.get_node_set_definition(circuit)}")

In [ ]:
# Difference (A - B)
combined_diff = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=ref_b, operation=SetOperation.DIFF
)
combined_diff.set_block_name("diff_ab")
print(f"Difference (A - B): {combined_diff.get_neuron_ids(circuit)}")
print(f"  Populations: {combined_diff.get_populations(circuit)}")
print(f"  Neuron IDs: {combined_diff.get_neuron_ids(circuit)}")
print(f"  Node set def: {combined_diff.get_node_set_definition(circuit)}")

#### (h) Recursive cycle detection in combined neuron sets

In [ ]:
# Create a recursive cycle: combined_x references combined_y, which references combined_x
combined_x = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=ref_b, operation=SetOperation.UNION
)
combined_x.set_block_name("combined_x")

combined_y = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=ref_b, operation=SetOperation.UNION
)
combined_y.set_block_name("combined_y")

# Now create the cycle: x references y, y references x
ref_x = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="combined_x")
ref_x.block = combined_x
ref_y = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="combined_y")
ref_y.block = combined_y

# Rewire to create the cycle
combined_x.combined_with = ref_y  # x combines with y
combined_y.combined_with = ref_x  # y combines with x (CYCLE!)

# This should raise a ValueError about recursive loop
try:
    combined_x.get_neuron_ids(circuit)
except ValueError as e:
    print(f"Caught expected error: {e}")

#### (i) CombinedNeuronSet — union kept symbolic

In [ ]:
# Create two base neuron sets to combine
nset_a = PredefinedPopulationNeuronSet(node_set="Layer3", population="S1nonbarrel_neurons")
nset_a.set_block_name("nset_a")

nset_b = PredefinedPopulationNeuronSet(node_set="Layer6", population="S1nonbarrel_neurons")
nset_b.set_block_name("nset_b")

# Wrap in BlockReferences (as required by the combined neuron set fields)
ref_a = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_a")
ref_a.block = nset_a
ref_b = BiophysicalNeuronSetReference(block_dict_name="neuron_sets", block_name="nset_b")
ref_b.block = nset_b

print(f"Set A ({nset_a.node_set}): {nset_a.get_node_set_definition(circuit)}")
print(f"Set B ({nset_b.node_set}): {nset_b.get_node_set_definition(circuit)}")

combined_union = BiophysicalCombinedNeuronSet(
    base_neuron_set=ref_a, combined_with=ref_b, operation=SetOperation.UNION
)
combined_union.set_block_name("union_ab")
print("Union:")
print(f"  Node set def: {combined_union.get_node_set_definition(circuit)}")

# Add to circuit and resolve
nset_name, sonata_circuit = combined_union.add_node_set_definition_to_sonata_circuit(circuit)
ntab = sonata_circuit.nodes["S1nonbarrel_neurons"].get(nset_name)


#### (j) AllNeurons — select all neurons (specific neuron sets)

In [ ]:
# AllNeurons — all neurons from all populations
nset = AllNeurons()
nset.set_block_name("all_neurons")

ids = nset.get_neuron_ids(circuit)
print(f"AllNeurons")
print(f"  Populations: {nset.get_populations(circuit)}")
print(f"  Neuron IDs: {ids}")

# Symbolic node set definition
nset_def, combined = nset.get_node_set_definition(circuit)
print(f"  Node set definition (symbolic): {nset_def}")
if combined:
    print(f"  Combined definitions: {combined}")

# Force resolve IDs
nset_def_resolved, combined_resolved = nset.get_node_set_definition(circuit, force_resolve_ids=True)
print(f"  Node set definition (resolved): {nset_def_resolved}")
if combined_resolved:
    print(f"  Combined definitions (resolved): {combined_resolved}")

In [ ]:
# AllBiophysicalNeurons — only biophysical populations
nset_bio = AllBiophysicalNeurons()
nset_bio.set_block_name("all_biophysical")

ids_bio = nset_bio.get_neuron_ids(circuit)
nset_def_bio, _ = nset_bio.get_node_set_definition(circuit)
print(f"AllBiophysicalNeurons")
print(f"  Populations: {nset_bio.get_populations(circuit)}")
print(f"  Neuron IDs: {ids_bio}")
print(f"  Node set definition (symbolic): {nset_def_bio}")

#### (k) Writing a node set to file

In [ ]:
import json

nset = PredefinedNeuronSet(node_set="Layer6")
nset.set_block_name("my_layer6_nset")

output_file = nset.to_node_set_file(
    circuit, output_path="./", file_name="neuron_set_2_test.json",
    overwrite_if_exists=True, init_empty=True
)
print(f"Written to: {output_file}")

with open(output_file) as f:
    print(json.dumps(json.load(f), indent=2))

# With force_resolve_ids=True
output_file_resolved = nset.to_node_set_file(
    circuit, output_path="./", file_name="neuron_set_2_test_resolved.json",
    overwrite_if_exists=True, init_empty=True, force_resolve_ids=True
)
print(f"\nWritten (resolved) to: {output_file_resolved}")
with open(output_file_resolved) as f:
    print(json.dumps(json.load(f), indent=2))

#### (l) Adding a node set definition to a SONATA circuit object

In [ ]:
# Create a neuron set and add it directly to the SONATA circuit object
nset = PropertyNeuronSet(
    population="S1nonbarrel_neurons",
    property_filter=NeuronPropertyFilter(filter_dict={"layer": ["6"], "synapse_class": ["EXC"]}),
)
nset.set_block_name("L6_EXC")

nset_name, sonata_circuit = nset.add_node_set_definition_to_sonata_circuit(circuit)
print(f"Added node set '{nset_name}' to SONATA circuit")
print(f"Node set content: {sonata_circuit.node_sets.content[nset_name]}")

# Verify it resolves correctly
resolved_ids = sonata_circuit.nodes['S1nonbarrel_neurons'].ids(nset_name)
print(f"Resolved IDs via SONATA: {resolved_ids}")